# Dia 05 — Requisições HTTP e APIs

**🃏 Cartas:** 5 · 26 · 47 &nbsp;·&nbsp; **📘 Original:** *Day 5. Handling HTTP requests and working with APIs*
&nbsp;·&nbsp; [repositório](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias) &nbsp;·&nbsp; [curso original do cipher387](https://github.com/cipher387/python-for-OSINT-21-days)

---

## 📖 Capítulo 5 — O endereço da loja

Segunda-feira, 9h15. Os depoimentos que você leu ontem estão impressos e grampeados sobre a mesa,
com marca-texto amarelo em cada chave PIX. A Iara pega a folha de cima e aponta o rodapé da loja,
copiado por uma das vítimas: **Av. Afonso Pena, 1500 — Centro — CEP 30130-010**.

— Golpista adora endereço. Endereço passa confiança. Mas CEP não mente: cada número aponta para um
lugar oficial, e existem APIs públicas que respondem isso em milissegundos.

Ela escreve duas URLs no quadro branco: **BrasilAPI** e **ViaCEP**.

— Você vai perguntar às duas. Se concordarem, ótimo. Se uma cair, a outra cobre. E repare: quando a
gente conversa com uma API, ela responde com um **status**, uns **cabeçalhos** e o **corpo**. Muita
gente só olha o corpo e perde metade da história.

Você roda o primeiro pedido. A resposta volta antes de você soltar a tecla Enter, e o logradouro
que a API devolve **não é** o que está escrito no rodapé. O CEP cai no coração do Centro de Belo
Horizonte: ponto de ônibus, comércio de rua, gente passando. Nada parecido com o galpão de onde
sairiam centenas de iPhones a preço de banana.

— Anota — diz a Iara. — "Endereço declarado não bate com o CEP." Não é prova, é pista. Mas é a
primeira inconsistência documentada da loja.

In [ ]:
#@title ⚙️ Preparar o ambiente (rode esta célula primeiro)
# No Colab: baixa o repositório e entra na pasta do dia.
# No Jupyter local: o notebook já está na pasta do dia, então não faz nada.
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias"
if "google.colab" in sys.modules:
    destino = "/content/python-para-osint-21-dias"
    if not os.path.isdir(destino):
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    os.chdir(f"{destino}/dia-05-http-e-apis")

PASTA_NB = str(Path.cwd())          # usada pelas células para voltar sempre à pasta certa
print("📂 Pasta do dia:", PASTA_NB)
print("🐍 Python", sys.version.split()[0])

## 🎯 Objetivos

- Entender o que acontece num pedido HTTP: **método**, **URL**, **status**, **cabeçalhos** e **corpo**.
- Rodar o exemplo original com `requests` e a API do GitHub.
- Fazer o mesmo pedido só com `urllib.request` (biblioteca padrão).
- Ler os cabeçalhos de **rate limit** e tratar os erros **404** e **429**.
- Consultar CEPs na **BrasilAPI** e no **ViaCEP** e comparar as duas fontes.

## 🧠 Conceito em 2 minutos

Quando o navegador abre uma página, ele manda um **pedido** (requisição) ao servidor e recebe uma
**resposta**. Um script faz exatamente a mesma coisa, só que sem janela.

| Parte | Exemplo | Para que serve |
|---|---|---|
| Método | `GET` | "me entregue isto" (o mais comum em OSINT) |
| URL | `https://viacep.com.br/ws/30130010/json/` | o endereço do recurso; parâmetros vêm depois do `?` |
| Status | `200`, `404`, `429` | 2xx deu certo · 4xx erro de quem pediu · 5xx erro do servidor |
| Cabeçalhos | `User-Agent`, `x-ratelimit-remaining` | metadados: quem pede, quanto ainda pode pedir… |
| Corpo | `{"logradouro": "…"}` | os dados em si (HTML, JSON, CSV…) |

Uma **API** é um "balcão de atendimento" para programas: em vez de uma página bonita, o servidor
devolve dados organizados, quase sempre em **JSON**. Muitas APIs públicas brasileiras não pedem
cadastro (BrasilAPI, ViaCEP, IBGE). Outras exigem uma **chave** e limitam quantos pedidos você faz
por hora: é o **rate limit**. Passou do limite, vem o **429 Too Many Requests**.

## 🥋 Nível 1 — Exercício original (cipher387)

| Arquivo | O que faz |
|---|---|
| [`original/api_request.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-05-http-e-apis/original/api_request.py) | Busca usuários com "javascript" na API do GitHub usando `requests` e imprime o JSON |

```bash
pip install requests
cd dia-05-http-e-apis
python original/api_request.py
```

> 💡 O autor recomenda testar APIs primeiro numa ferramenta online (como o reqbin.com) antes de
> escrever código. Vale a dica: você enxerga status e cabeçalhos sem programar nada. Ele também mantém
> uma lista com mais de cem APIs úteis para OSINT: <https://github.com/cipher387/API-s-for-OSINT>.

In [ ]:
#@title 📦 Instalar as bibliotecas do exercício original
%pip install -q requests

### `original/api_request.py` — Primeiro pedido a uma API (busca de usuários no GitHub)

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Primeiro pedido a uma API (busca de usuários no GitHub)
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_5/api_request.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# Requer: pip install requests
# ──────────────────────────────────────────────────────────────────────────

# 💬 "requests" é o pacote mais famoso para fazer pedidos HTTP em Python.
# 💬 Ele não vem com o Python: precisa do pip install acima.
import requests

# 💬 GET = "me entregue esta página". A URL é a API de busca de usuários do GitHub;
# 💬 tudo depois do "?" é parâmetro: q=javascript significa "procure por javascript".
response = requests.get("https://api.github.com/search/users?q=javascript")

# 💬 A API responde em JSON; .json() transforma esse texto em dicionário Python.
# 💬 O print mostra tudo de uma vez (é muita coisa!). No Dia 6 aprendemos a escolher campos.
print(response.json())

## 🇧🇷 Nível 2 — Camada Brasil

Tudo com a **biblioteca padrão**. Cada script tem amostra offline em `dados/`.

| Arquivo | O que faz |
|---|---|
| [`br/01_github_stdlib.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-05-http-e-apis/br/01_github_stdlib.py) | O mesmo pedido do original com `urllib`, mostrando status e cabeçalhos de rate limit |
| [`br/02_cep_brasilapi.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-05-http-e-apis/br/02_cep_brasilapi.py) | Consulta um CEP na BrasilAPI v2 e lê os campos aninhados (coordenadas) |
| [`br/03_viacep.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-05-http-e-apis/br/03_viacep.py) | Consulta o ViaCEP, compara com a BrasilAPI e trata CEP inexistente (`{"erro": "true"}`) |
| [`br/04_caso_cep_rodape.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-05-http-e-apis/br/04_caso_cep_rodape.py) | **O caso:** extrai o CEP do rodapé da vitrine, trata 404/429 e cruza com o que a loja promete |

```bash
cd dia-05-http-e-apis
python br/01_github_stdlib.py
OSINT_OFFLINE=1 python br/04_caso_cep_rodape.py     # força o modo offline
```

**O que você deve ver no caso:** o script compara o logradouro que a loja **declara** no rodapé
("Av. Afonso Pena, 1500") com o logradouro que a API **devolve** para o CEP `30130-010`. Na amostra
offline (montada à mão, não baixada ao vivo) o CEP aparece como **Praça Sete de Setembro**, no Centro
de BH. Rode com internet e confira o que a API responde hoje: se o logradouro for outro que não o
declarado, essa **divergência** é a pista do capítulo, e vai para o relatório com a fonte e a hora.

> ⚠️ **Armadilha clássica:** o ViaCEP responde **HTTP 200** mesmo quando o CEP não existe; o aviso vem
> no corpo, como `{"erro": "true"}` (versões antigas mandavam o booleano `true`). Já a BrasilAPI
> responde **404**. Duas APIs, dois jeitos de dizer "não achei": seu código precisa entender os dois.

> 💡 Os campos com formulário à direita (`#@param`) podem ser editados sem mexer no código. Para testar sem internet, rode antes: `os.environ['OSINT_OFFLINE'] = '1'`.

### `br/01_github_stdlib.py` — O mesmo pedido do original (busca de usuários no GitHub), só com a biblioteca padrão.

In [ ]:
%cd -q {PASTA_NB}
"""
O mesmo pedido do original (busca de usuários no GitHub), só com a biblioteca padrão.

Carta 26 (rank B) — Jutsu do Mensageiro HTTP

O que você aprende:
  • a fazer um GET com urllib.request (vem com o Python, sem pip install);
  • a ler as três partes de uma resposta HTTP: status, cabeçalhos e corpo;
  • a ler os cabeçalhos de limite de uso (rate limit) que o GitHub manda;
  • a montar parâmetros de URL com urllib.parse.urlencode (acentos e espaços seguros).

Como rodar (dentro da pasta dia-05-http-e-apis):
    python br/01_github_stdlib.py
    OSINT_OFFLINE=1 python br/01_github_stdlib.py   # sem internet, usa dados/
"""

import json                 # transformar o texto JSON em dicionário Python
import os                   # ler a variável de ambiente OSINT_OFFLINE
import urllib.parse         # montar a parte "?q=..." da URL com segurança
import urllib.request       # fazer o pedido HTTP
from datetime import datetime, timedelta, timezone  # converter o horário do reset
from pathlib import Path    # montar caminhos de arquivo

# ── Parâmetros (no Colab viram um formulário) ─────────────────────────────
BUSCA = "osint"  #@param {type:"string"}
QUANTOS = 5  #@param {type:"integer"}

# ── Pastas e modo offline ─────────────────────────────────────────────────
PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
OFFLINE = os.environ.get("OSINT_OFFLINE") == "1"

# Horário de Brasília. No Windows pode faltar o banco de fusos (tzdata): usamos -3h fixo.
try:
    from zoneinfo import ZoneInfo
    FUSO_BR = ZoneInfo("America/Sao_Paulo")
except Exception:
    FUSO_BR = timezone(timedelta(hours=-3))


def pedir(url):
    """Faz um GET e devolve (status, cabeçalhos, corpo_em_dicionário)."""
    pedido = urllib.request.Request(url, headers={
        # O GitHub EXIGE um User-Agent. Sem ele, a API recusa o pedido.
        "User-Agent": "python-para-osint-21-dias/estudo",
        # Pedimos explicitamente a versão JSON da API.
        "Accept": "application/vnd.github+json",
    })
    with urllib.request.urlopen(pedido, timeout=15) as resposta:
        status = resposta.status                       # 200 = deu certo
        cabecalhos = {k.lower(): v for k, v in resposta.headers.items()}
        corpo = json.load(resposta)                    # lê e converte o JSON
    return status, cabecalhos, corpo


# ── 1. Montar a URL ───────────────────────────────────────────────────────
# urlencode cuida de espaços e acentos: "são paulo" vira "s%C3%A3o+paulo".
parametros = urllib.parse.urlencode({"q": BUSCA, "per_page": QUANTOS})
url = "https://api.github.com/search/users?" + parametros
print(f"🌐 GET {url}\n")

# ── 2. Fazer o pedido (ou usar a amostra) ─────────────────────────────────
resultado = None
if not OFFLINE:
    try:
        resultado = pedir(url)
    except Exception as erro:
        print(f"⚠️ MODO OFFLINE: a consulta falhou ({erro}). Usando a amostra local.")
else:
    print("⚠️ MODO OFFLINE: usando a amostra local.")

if resultado is None:
    amostra = json.loads((PASTA_DIA / "dados" / "github_search_users.json").read_text(encoding="utf-8"))
    # A amostra guarda status e cabeçalhos em campos extras que começam com "_".
    resultado = (amostra["_status"], amostra["_cabecalhos"], amostra)

status, cabecalhos, corpo = resultado

# ── 3. Status: o "deu certo?" do HTTP ─────────────────────────────────────
# 2xx = sucesso · 3xx = redirecionamento · 4xx = erro seu · 5xx = erro do servidor
print(f"📶 Status HTTP: {status}")

# ── 4. Cabeçalhos de rate limit ───────────────────────────────────────────
# O GitHub conta quantos pedidos você fez. A busca tem limite bem menor que o resto.
print("\n⏱️  Limite de uso (rate limit):")
for nome in ["x-ratelimit-resource", "x-ratelimit-limit", "x-ratelimit-used", "x-ratelimit-remaining"]:
    print(f"   {nome:<22} = {cabecalhos.get(nome, '(não veio)')}")

reset = cabecalhos.get("x-ratelimit-reset")
if reset:
    # O reset vem como "timestamp Unix" (segundos desde 01/01/1970, em UTC).
    quando = datetime.fromtimestamp(int(reset), tz=timezone.utc).astimezone(FUSO_BR)
    print(f"   o contador zera em    = {quando:%d/%m/%Y %H:%M} (horário de Brasília)")

# ── 5. Corpo: o JSON com os resultados ────────────────────────────────────
print(f"\n🔢 total_count = {corpo.get('total_count')}")
for usuario in corpo.get("items", [])[:QUANTOS]:
    print(f"   👤 {usuario.get('login'):<22} {usuario.get('type', ''):<13} {usuario.get('html_url')}")

print("\n💡 Compare com original/api_request.py: lá o requests fez tudo isso escondido.")

### `br/02_cep_brasilapi.py` — Consultar um CEP na BrasilAPI (uma API brasileira, gratuita e sem cadastro).

In [ ]:
%cd -q {PASTA_NB}
"""
Consultar um CEP na BrasilAPI (uma API brasileira, gratuita e sem cadastro).

Carta 26 (rank B) — Jutsu do Mensageiro HTTP (parte 2)

O que você aprende:
  • a consultar uma API pública brasileira só com a biblioteca padrão;
  • a ler campos aninhados (location → coordinates → latitude);
  • a limpar a entrada (tirar ponto e traço do CEP) antes de montar a URL;
  • a usar .get() para não quebrar quando um campo não vier.

Como rodar (dentro da pasta dia-05-http-e-apis):
    python br/02_cep_brasilapi.py
    OSINT_OFFLINE=1 python br/02_cep_brasilapi.py
"""

import json
import os
import urllib.request
from pathlib import Path

# ── Parâmetros ────────────────────────────────────────────────────────────
CEP = "30130-010"  #@param {type:"string"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
OFFLINE = os.environ.get("OSINT_OFFLINE") == "1"


def buscar_json(url, amostra):
    """Baixa um JSON da internet; se falhar (ou OFFLINE=1), lê a amostra local."""
    if not OFFLINE:
        try:
            pedido = urllib.request.Request(url, headers={"User-Agent": "python-para-osint-21-dias/estudo"})
            with urllib.request.urlopen(pedido, timeout=15) as resposta:
                return json.load(resposta)
        except Exception as erro:
            print(f"⚠️ MODO OFFLINE: a consulta falhou ({erro}). Usando a amostra local.")
    else:
        print("⚠️ MODO OFFLINE: usando a amostra local.")
    return json.loads((PASTA_DIA / "dados" / amostra).read_text(encoding="utf-8"))


# ── 1. Limpar o CEP: só os 8 dígitos ──────────────────────────────────────
# "30130-010" → "30130010". Isso evita URL errada e também protege contra
# alguém colar texto estranho no parâmetro.
cep = "".join(c for c in CEP if c.isdigit())
if len(cep) != 8:
    raise SystemExit(f"CEP precisa ter 8 dígitos; recebi {CEP!r}")

# ── 2. Consultar ──────────────────────────────────────────────────────────
# A v2 da BrasilAPI pergunta a vários provedores e devolve o primeiro que responder.
url = f"https://brasilapi.com.br/api/cep/v2/{cep}"
print(f"🌐 GET {url}\n")
dados = buscar_json(url, "brasilapi_cep_30130010.json")

# ── 3. Mostrar os campos principais ───────────────────────────────────────
# Os nomes dos campos estão em inglês (é o padrão da BrasilAPI).
print(f"📮 CEP:        {dados.get('cep')}")
print(f"🛣️  Logradouro: {dados.get('street') or '(sem logradouro: CEP geral da cidade?)'}")
print(f"🏘️  Bairro:     {dados.get('neighborhood')}")
print(f"🏙️  Cidade/UF:  {dados.get('city')}/{dados.get('state')}")
print(f"🛰️  Fonte:      {dados.get('service')}  ← qual provedor respondeu de verdade")

# ── 4. Campo aninhado: coordenadas ────────────────────────────────────────
# location é um dicionário que tem outro dicionário dentro (coordinates).
# Encadeamos .get(..., {}) para não quebrar se algum nível estiver vazio.
coordenadas = dados.get("location", {}).get("coordinates", {})
lat, lon = coordenadas.get("latitude"), coordenadas.get("longitude")
if lat and lon:
    print(f"📍 Coordenadas: {lat}, {lon}")
    print(f"🗺️  Ver no mapa: https://www.openstreetmap.org/?mlat={lat}&mlon={lon}#map=18/{lat}/{lon}")
else:
    print("📍 Coordenadas: não vieram (acontece com vários CEPs; não é erro).")

### `br/03_viacep.py` — ViaCEP x BrasilAPI: duas fontes para o mesmo CEP (e o que fazer quando o CEP não existe).

In [ ]:
%cd -q {PASTA_NB}
"""
ViaCEP x BrasilAPI: duas fontes para o mesmo CEP (e o que fazer quando o CEP não existe).

Carta 26 (rank B) — Jutsu do Mensageiro HTTP (parte 3)

O que você aprende:
  • a consultar o ViaCEP, outra API pública de CEP;
  • que "CEP inexistente" no ViaCEP NÃO é erro HTTP: vem status 200 com {"erro": "true"};
  • a tratar o campo "erro" tanto como texto ("true") quanto como booleano (true);
  • a comparar duas fontes campo a campo (regra de ouro do OSINT: confirmar).

Como rodar (dentro da pasta dia-05-http-e-apis):
    python br/03_viacep.py
    OSINT_OFFLINE=1 python br/03_viacep.py
"""

import json
import os
import unicodedata          # comparar textos ignorando acentos
import urllib.request
from pathlib import Path

# ── Parâmetros ────────────────────────────────────────────────────────────
CEP = "30130-010"  #@param {type:"string"}
CEP_INEXISTENTE = "99999-999"  #@param {type:"string"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
OFFLINE = os.environ.get("OSINT_OFFLINE") == "1"


def buscar_json(url, amostra):
    """Baixa um JSON da internet; se falhar (ou OFFLINE=1), lê a amostra local."""
    if not OFFLINE:
        try:
            pedido = urllib.request.Request(url, headers={"User-Agent": "python-para-osint-21-dias/estudo"})
            with urllib.request.urlopen(pedido, timeout=15) as resposta:
                return json.load(resposta)
        except Exception as erro:
            print(f"⚠️ MODO OFFLINE: a consulta falhou ({erro}). Usando a amostra local.")
    else:
        print("⚠️ MODO OFFLINE: usando a amostra local.")
    return json.loads((PASTA_DIA / "dados" / amostra).read_text(encoding="utf-8"))


def so_digitos(texto):
    """'30130-010' → '30130010'."""
    return "".join(c for c in texto if c.isdigit())


def cep_nao_existe(resposta):
    """O ViaCEP sinaliza CEP inexistente com "erro": "true" (texto) ou true (booleano)."""
    return str(resposta.get("erro", "")).lower() == "true"


def normalizar(texto):
    """Tira acentos e deixa minúsculo, para comparar 'Praça' com 'Praca'."""
    sem_acento = unicodedata.normalize("NFKD", texto or "").encode("ascii", "ignore").decode()
    return sem_acento.casefold().strip()


# ── 1. Consulta no ViaCEP ─────────────────────────────────────────────────
cep = so_digitos(CEP)
via = buscar_json(f"https://viacep.com.br/ws/{cep}/json/", "viacep_30130010.json")
if cep_nao_existe(via):
    raise SystemExit(f"O ViaCEP diz que {CEP} não existe.")
print(f"📮 ViaCEP:    {via.get('logradouro')} · {via.get('bairro')} · {via.get('localidade')}/{via.get('uf')}")
print(f"   extras:    código IBGE {via.get('ibge')} · DDD {via.get('ddd')}")

# ── 2. Mesma pergunta à BrasilAPI ─────────────────────────────────────────
bra = buscar_json(f"https://brasilapi.com.br/api/cep/v2/{cep}", "brasilapi_cep_30130010.json")
print(f"📮 BrasilAPI: {bra.get('street')} · {bra.get('neighborhood')} · {bra.get('city')}/{bra.get('state')}")

# ── 3. Comparar campo a campo ─────────────────────────────────────────────
# Os nomes dos campos são diferentes em cada API; este dicionário faz a "tradução".
pares = {"logradouro": "street", "bairro": "neighborhood", "localidade": "city", "uf": "state"}
print("\n🔍 Comparação:")
divergencias = 0
for campo_via, campo_bra in pares.items():
    a, b = via.get(campo_via, ""), bra.get(campo_bra, "")
    igual = normalizar(a) == normalizar(b)
    divergencias += not igual          # True conta como 1, False como 0
    print(f"   {'✅' if igual else '🚩'} {campo_via:<11} ViaCEP={a!r:<28} BrasilAPI={b!r}")
print("   → As fontes concordam." if divergencias == 0 else f"   → {divergencias} divergência(s): anote e investigue.")

# ── 4. E quando o CEP não existe? ─────────────────────────────────────────
print(f"\n🧪 Testando um CEP que não existe: {CEP_INEXISTENTE}")
falso = buscar_json(f"https://viacep.com.br/ws/{so_digitos(CEP_INEXISTENTE)}/json/", "viacep_inexistente.json")
if cep_nao_existe(falso):
    print("   ❌ ViaCEP respondeu {'erro': 'true'}: o pedido deu certo (HTTP 200), mas o CEP não existe.")
    print("   💡 Por isso não basta checar o status HTTP: leia também o corpo da resposta.")
else:
    print(f"   Resposta: {falso}")

### `br/04_caso_cep_rodape.py` — Caso Vitrine Fantasma: o CEP do rodapé da loja leva a um depósito?

In [ ]:
%cd -q {PASTA_NB}
"""
Caso Vitrine Fantasma: o CEP do rodapé da loja leva a um depósito?

Carta 26 (rank B) — Jutsu do Mensageiro HTTP (aplicação no caso)

O que você aprende:
  • a extrair o CEP do rodapé da cópia salva do site (caso/dados/vitrine_megaoferta.html);
  • a tratar os erros HTTP mais comuns de APIs: 404 (não encontrado) e 429 (pediu demais);
  • a respeitar o cabeçalho Retry-After e tentar de novo com espera crescente;
  • a cruzar o endereço oficial do CEP com o que a loja promete.

Como rodar (dentro da pasta dia-05-http-e-apis):
    python br/04_caso_cep_rodape.py
    OSINT_OFFLINE=1 python br/04_caso_cep_rodape.py
"""

import json
import os
import re                   # expressão regular para achar o CEP no HTML (mais no Dia 11)
import time                 # esperar entre tentativas
import urllib.error         # os tipos de erro HTTP (HTTPError, URLError)
import urllib.request
from pathlib import Path

# ── Parâmetros ────────────────────────────────────────────────────────────
TENTATIVAS = 3  #@param {type:"integer"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"
OFFLINE = os.environ.get("OSINT_OFFLINE") == "1"


def pedir_com_paciencia(url, tentativas=TENTATIVAS):
    """GET que entende 404 e 429. Devolve o dicionário, ou None se o recurso não existe."""
    pedido = urllib.request.Request(url, headers={"User-Agent": "python-para-osint-21-dias/estudo"})
    for tentativa in range(1, tentativas + 1):
        try:
            with urllib.request.urlopen(pedido, timeout=15) as resposta:
                return json.load(resposta)
        except urllib.error.HTTPError as erro:
            if erro.code == 404:
                # 404 = "não encontrei". Para CEP, significa CEP inexistente: não adianta insistir.
                print("   ❌ HTTP 404: o servidor diz que esse CEP não existe.")
                return None
            if erro.code == 429 and tentativa < tentativas:
                # 429 = "muitos pedidos". O servidor pode dizer quanto esperar (Retry-After).
                espera = int(erro.headers.get("Retry-After") or 2 ** tentativa)
                print(f"   ⏳ HTTP 429: pedimos rápido demais. Esperando {espera}s (tentativa {tentativa}).")
                time.sleep(espera)
                continue
            raise                       # outros erros sobem para quem chamou
    return None


# ── 1. Achar o CEP no rodapé da vitrine salva ─────────────────────────────
html = (PASTA_CASO / "vitrine_megaoferta.html").read_text(encoding="utf-8")
# Padrão: 5 dígitos, hífen opcional, 3 dígitos. O Dia 11 explica regex em detalhe.
achado = re.search(r"CEP\s*(\d{5}-?\d{3})", html)
cep_rodape = achado.group(1) if achado else "30130-010"
# Também pegamos a linha de endereço inteira para comparar depois.
linha_endereco = next((linha.strip() for linha in html.splitlines() if "CEP" in linha), "")
linha_endereco = re.sub(r"<[^>]+>", "", linha_endereco)      # tira as tags <p>...</p>
print(f"🏷️  Rodapé da loja: {linha_endereco}")
print(f"📮 CEP encontrado: {cep_rodape}\n")

# ── 2. Consultar o CEP (BrasilAPI) ────────────────────────────────────────
cep = cep_rodape.replace("-", "")
url = f"https://brasilapi.com.br/api/cep/v2/{cep}"
dados = None
if not OFFLINE:
    try:
        dados = pedir_com_paciencia(url)
    except Exception as erro:
        print(f"⚠️ MODO OFFLINE: a consulta falhou ({erro}). Usando a amostra local.")
else:
    print("⚠️ MODO OFFLINE: usando a amostra local.")
if dados is None:
    dados = json.loads((PASTA_DIA / "dados" / "brasilapi_cep_30130010.json").read_text(encoding="utf-8"))

# ── 3. O que é esse endereço de verdade? ──────────────────────────────────
oficial = f"{dados.get('street')}, {dados.get('neighborhood')}, {dados.get('city')}/{dados.get('state')}"
print(f"🗺️  Endereço oficial do CEP: {oficial}")

# ── 4. Cruzar o que o SITE DIZ com o que a API RESPONDE ───────────────────
# Não decidimos de antemão o que a API "deveria" dizer: comparamos o que veio.
def simplificar(texto):
    """'Av. Afonso Pena' → 'afonso pena' (tira tipo de logradouro, acento e caixa)."""
    import unicodedata
    t = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode().casefold()
    t = re.sub(r"^(av\.?|avenida|r\.?|rua|pca\.?|praca|al\.?|alameda)\s+", "", t.strip())
    return " ".join(t.split())

rua_no_rodape = linha_endereco.split(",")[0].split("—")[0].strip()   # ex.: "Av. Afonso Pena"
rua_da_api = dados.get("street") or ""
print("\n🔍 Cruzamento (declarado x oficial):")
print(f"   • O site declara:  {rua_no_rodape!r}")
print(f"   • O CEP pertence a: {rua_da_api or '(API não informou logradouro)'!r}")
if rua_da_api and simplificar(rua_no_rodape) != simplificar(rua_da_api):
    print("   🚩 DIVERGÊNCIA: o logradouro do CEP não é o logradouro que a loja escreveu.")
    print("      CEPs de centro de cidade às vezes cobrem praças e cruzamentos, então isso é PISTA,")
    print("      não prova. Mas é exatamente o tipo de detalhe que golpista não confere.")
    conclusao = "Endereço declarado no rodapé não corresponde ao logradouro do CEP informado."
elif rua_da_api:
    print("   ✅ O logradouro bate. Ainda assim: o endereço tem cara de depósito de eletrônicos?")
    conclusao = "Logradouro confere com o CEP; verificar se o local é compatível com a operação."
else:
    conclusao = "API sem logradouro para o CEP; consultar outra fonte (ViaCEP)."
print(f"   • Bairro/cidade do CEP: {dados.get('neighborhood')}, {dados.get('city')}/{dados.get('state')}")

print(f"\n📝 Para o relatório: '{conclusao}'")
print("   Registre também a fonte (campo 'service' da BrasilAPI) e a data/hora da consulta.")
print("   Próximo passo sugerido pela Iara: olhar o local no Street View (sem sair da sala!).")

## ⚖️ Ética, LGPD e OPSEC

- **CEP não é dado pessoal**, mas CEP + número + nome de uma pessoa é. Ao consultar endereços de
  vítimas, guarde só o necessário para a finalidade do caso (LGPD, art. 6º: finalidade e necessidade).
- **Rate limit é um contrato.** Não rode laços sem pausa contra APIs gratuitas: você derruba um serviço
  que todo o país usa e ainda pode ter seu IP bloqueado. Use `time.sleep` e respeite o `Retry-After`.
- **Toda consulta deixa rastro.** O servidor da API registra seu IP, horário e o que você perguntou.
  APIs públicas de CEP são inofensivas, mas uma API que **pertence ao alvo** (o site do golpista, por
  exemplo) vê você chegando. Isso é assunto do Dia 12.
- **Leia os termos de uso.** Algumas APIs proíbem uso comercial ou armazenamento em massa.
- **Nunca coloque chaves de API no código** que vai para o GitHub. Use variáveis de ambiente.

## 🎯 Quiz rápido

Escolha a resposta no formulário de cada célula e rode a célula.

In [ ]:
#@title 1. O ViaCEP respondeu HTTP 200 com o corpo {'erro': 'true'}. O que isso significa?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) O servidor caiu", "B) O pedido funcionou, mas o CEP não existe", "C) Você precisa de uma chave de API", "D) O CEP existe, mas está sem logradouro"]
correta = "B) O pedido funcionou, mas o CEP não existe"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "O ViaCEP sinaliza CEP inexistente no corpo da resposta, não no status. Por isso é preciso ler o corpo, e não só o código HTTP.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 2. Qual código HTTP indica que você fez pedidos demais em pouco tempo?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) 200", "B) 301", "C) 404", "D) 429"]
correta = "D) 429"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "429 Too Many Requests. O certo é esperar (de preferência o tempo do cabeçalho Retry-After) antes de tentar de novo.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 3. Para que serve o cabeçalho x-ratelimit-remaining que o GitHub devolve?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) Mostra quantos pedidos ainda restam na janela atual", "B) Mostra quantos usuários existem no GitHub", "C) Indica a velocidade da sua internet", "D) Informa a senha temporária da API"]
correta = "A) Mostra quantos pedidos ainda restam na janela atual"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "Os cabeçalhos x-ratelimit-* dizem o limite, quanto já foi usado, quanto resta e quando o contador zera (x-ratelimit-reset).")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

# 🏆 Desafio S — Dia 5: Verificador de CEPs em lote

**Carta 47 · Rank S**

## A missão

As denúncias continuam chegando, e muitas trazem um "endereço de retirada" ou de "devolução" que o
vendedor passou pelo WhatsApp. A Iara quer conferir **todos de uma vez**, sem depender de uma única API.

Escreva um script que:

1. Receba uma lista de CEPs separados por vírgula (parâmetro `CEPS`), com ou sem hífen.
2. **Não consulte** o que não tem 8 dígitos (marque "formato inválido").
3. Pergunte primeiro à **BrasilAPI**; se ela falhar ou não achar, pergunte ao **ViaCEP** (fonte reserva).
4. Reconheça CEP inexistente nas **duas** formas: HTTP 404 (BrasilAPI) e `{"erro": "true"}` (ViaCEP).
5. Registre de **qual fonte** veio cada resposta.
6. Espere um pouco entre consultas (`PAUSA_SEGUNDOS`) para não levar um 429.
7. Grave `saidas/ceps_verificados.csv` com `;` e `utf-8-sig`.
8. Funcione **sem internet** com as amostras de `dados/`.

## Critérios de rank S

- [ ] Só biblioteca padrão
- [ ] Um CEP com problema não derruba o lote
- [ ] O CEP `99999-999` sai como 🚩 inexistente, e `1234` sai como formato inválido sem consulta
- [ ] A coluna `fonte` mostra quando o ViaCEP salvou o dia

## Dica

Faça cada fonte virar uma função que devolve **duas coisas**: os dados (ou `None`) e uma situação
(`"ok"`, `"inexistente"`, `"falhou"`). Assim o laço principal fica curto e fácil de ler.

In [ ]:
%cd -q {PASTA_NB}
# ✍️ Escreva sua solução aqui.

In [ ]:
#@title 🔒 Gabarito (dê dois cliques aqui para ver o código) { display-mode: "form" }
%cd -q {PASTA_NB}
"""
Gabarito do Desafio S do Dia 5 — Verificador de CEPs em lote com fonte reserva.

Recebe vários CEPs, consulta a BrasilAPI e, se ela falhar, pergunta ao ViaCEP.
Marca CEPs inexistentes, mostra de qual fonte veio cada resposta e grava
saidas/ceps_verificados.csv. Sem internet (ou com OSINT_OFFLINE=1), usa as amostras.
"""

import csv
import json
import os
import time
import urllib.error
import urllib.request
from pathlib import Path

CEPS = "30130-010, 01001-000, 99999-999, 1234"  #@param {type:"string"}
PAUSA_SEGUNDOS = 1  #@param {type:"number"}

PASTA_DIA = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
OFFLINE = os.environ.get("OSINT_OFFLINE") == "1"

# Amostras locais por CEP (só existem para alguns; os outros ficam "sem dados" offline)
AMOSTRAS_BRASILAPI = {"30130010": "brasilapi_cep_30130010.json"}
AMOSTRAS_VIACEP = {"30130010": "viacep_30130010.json", "01001000": "viacep_01001000.json",
                   "99999999": "viacep_inexistente.json"}


def baixar(url):
    pedido = urllib.request.Request(url, headers={"User-Agent": "python-para-osint-21-dias/estudo"})
    with urllib.request.urlopen(pedido, timeout=15) as resposta:
        return json.load(resposta)


def amostra(nome):
    return json.loads((PASTA_DIA / "dados" / nome).read_text(encoding="utf-8"))


def via_brasilapi(cep):
    """Devolve (linha, situação). Situação: 'ok', 'inexistente' ou 'falhou'."""
    if OFFLINE:
        if cep in AMOSTRAS_BRASILAPI:
            d = amostra(AMOSTRAS_BRASILAPI[cep])
        else:
            return None, "falhou"
    else:
        try:
            d = baixar(f"https://brasilapi.com.br/api/cep/v2/{cep}")
        except urllib.error.HTTPError as erro:
            return None, "inexistente" if erro.code == 404 else "falhou"
        except Exception:
            return None, "falhou"
    return {"logradouro": d.get("street", ""), "bairro": d.get("neighborhood", ""),
            "cidade": d.get("city", ""), "uf": d.get("state", "")}, "ok"


def via_viacep(cep):
    if OFFLINE:
        if cep not in AMOSTRAS_VIACEP:
            return None, "falhou"
        d = amostra(AMOSTRAS_VIACEP[cep])
    else:
        try:
            d = baixar(f"https://viacep.com.br/ws/{cep}/json/")
        except Exception:
            return None, "falhou"
    if str(d.get("erro", "")).lower() == "true":      # "true" (texto) ou true (booleano)
        return None, "inexistente"
    return {"logradouro": d.get("logradouro", ""), "bairro": d.get("bairro", ""),
            "cidade": d.get("localidade", ""), "uf": d.get("uf", "")}, "ok"


if OFFLINE:
    print("⚠️ MODO OFFLINE: usando as amostras locais.")

linhas = []
for bruto in [c.strip() for c in CEPS.split(",") if c.strip()]:
    cep = "".join(ch for ch in bruto if ch.isdigit())
    linha = {"cep_digitado": bruto, "cep": cep, "fonte": "", "situacao": "",
             "logradouro": "", "bairro": "", "cidade": "", "uf": ""}
    if len(cep) != 8:
        linha["situacao"] = "formato inválido (não consultado)"   # não gasta consulta à toa
    else:
        dados, situacao = via_brasilapi(cep)
        fonte = "BrasilAPI"
        if situacao != "ok":                                       # fonte reserva
            dados, situacao = via_viacep(cep)
            fonte = "ViaCEP"
        linha["fonte"] = fonte if situacao != "falhou" else ""
        linha["situacao"] = {"ok": "encontrado", "inexistente": "🚩 CEP inexistente",
                             "falhou": "sem resposta"}[situacao]
        if dados:
            linha.update(dados)
        if not OFFLINE:
            time.sleep(PAUSA_SEGUNDOS)                             # educação com a API
    linhas.append(linha)
    print(f"{bruto:<11} {linha['situacao']:<34} {linha['fonte']:<10} {linha['logradouro']} {linha['cidade']}")

SAIDAS = PASTA_DIA / "saidas"
SAIDAS.mkdir(exist_ok=True)
with open(SAIDAS / "ceps_verificados.csv", "w", newline="", encoding="utf-8-sig") as f:
    escritor = csv.DictWriter(f, fieldnames=list(linhas[0]), delimiter=";")
    escritor.writeheader()
    escritor.writerows(linhas)
print("\n💾 saidas/ceps_verificados.csv gravado.")

## 🔎 Inovação OSINT do dia

**BrasilAPI.** Projeto aberto, mantido pela comunidade, que reúne várias fontes públicas brasileiras
numa API só e gratuita: CEP, CNPJ, bancos, DDD, feriados nacionais, tabela FIPE, entre outras. A rota
de CEP v2 pergunta a mais de um provedor e devolve, no campo `service`, **qual deles respondeu**:
ótimo para registrar a origem da informação no relatório. Biblioteca Python simples: `urllib.request` +
`json` (nativos). Amanhã usamos a mesma API para consultar CNPJ.

## 📚 Para ir além

- BrasilAPI (documentação): <https://brasilapi.com.br/docs>
- ViaCEP (documentação): <https://viacep.com.br/>
- Limites de uso da API REST do GitHub: <https://docs.github.com/pt/rest/using-the-rest-api/rate-limits-for-the-rest-api>
- Códigos de status HTTP explicados (MDN, em português): <https://developer.mozilla.org/pt-BR/docs/Web/HTTP/Status>
- Lista de APIs para OSINT do cipher387: <https://github.com/cipher387/API-s-for-OSINT>

## ✅ Checklist

- [ ] Rodei o script original com `requests`
- [ ] Fiz o mesmo pedido com `urllib` e li os cabeçalhos de rate limit
- [ ] Consultei um CEP na BrasilAPI e no ViaCEP e comparei
- [ ] Vi a diferença entre "HTTP 404" e "HTTP 200 com erro no corpo"
- [ ] Descobri o que é o endereço do rodapé da loja
- [ ] Fiz o desafio S
- [ ] Anotei no meu [`DIARIO.md`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/DIARIO.md)

---

🎉 Terminou? [Registre sua entrega do Dia 5](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/issues/new?template=entreguei-o-dia.yml) · Baseado em [cipher387/python-for-OSINT-21-days](https://github.com/cipher387/python-for-OSINT-21-days) (MIT) · Adaptação pt-BR: [Ridd1kulusC0d3r/python-para-osint-21-dias](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias)